# 42 · JDBC round trip and partitioned reads

**Scenario:** Optionally export the asset slice to a dedicated JDBC lab table, then read that same table back.

**Source:** Databricks-provided `samples.nyctaxi.trips` only. File formats, nested structures, replays and histories in this lab are derived from that asset. No hand-created record dataset, third-party download, pandas or generated replacement data is used.

**Prerequisites:** Run notebook 00 first. Each notebook is independent; run its cells in order. Mode: **integration**. Disabled by default; JDBC driver/network/secret scope and dedicated database table required.

**Learning-size sample:** `max_rows` defaults to 2,000. Sorting/limiting caps result size, not the underlying scan cost. Assertions use relationships instead of hardcoded dataset counts because sample assets may change.


## Step 1 — Load and inspect the Databricks asset

Read the built-in sample and normalize column names. A stable content fingerprint helps with replay examples; it is not a guaranteed real-world trip key. The original `samples` asset is always read-only.


In [ ]:
from pyspark.sql import functions as F, types as T, Window
dbutils.widgets.text('max_rows','2000')
MAX_ROWS=int(dbutils.widgets.get('max_rows'))
assert 100<=MAX_ROWS<=20000, 'Use 100–20000 rows for these bounded learning labs.'
SOURCE_ASSET='samples.nyctaxi.trips'
try:
    asset=(spark.read.table(SOURCE_ASSET)
        .select('tpep_pickup_datetime','tpep_dropoff_datetime','trip_distance',
                'fare_amount','pickup_zip','dropoff_zip')
        .orderBy('tpep_pickup_datetime','tpep_dropoff_datetime','pickup_zip',
                 'dropoff_zip','trip_distance','fare_amount')
        .limit(MAX_ROWS))
    assert asset.limit(1).count()==1, 'Databricks sample asset is empty.'
except Exception as exc:
    raise RuntimeError('This course requires the Databricks-provided samples.nyctaxi.trips asset. '
        'Check Unity Catalog/sample access on the attached compute. No synthetic or external fallback is used.') from exc
df=(asset.select(F.col('tpep_pickup_datetime').cast('timestamp').alias('pickup_ts'),
    F.col('tpep_dropoff_datetime').cast('timestamp').alias('dropoff_ts'),
    F.col('trip_distance').cast('double').alias('distance'),
    F.col('fare_amount').cast('double').alias('fare'),
    F.col('pickup_zip').cast('string').alias('pickup_zip'),
    F.col('dropoff_zip').cast('string').alias('dropoff_zip')))
df=df.withColumn('trip_id',F.sha2(F.to_json(F.struct(*[F.col(c) for c in df.columns]),
    options={'ignoreNullFields':'false'}),256))
df=df.withColumn('pickup_date',F.to_date('pickup_ts'))
df=df.withColumn('duration_sec',F.unix_timestamp('dropoff_ts')-F.unix_timestamp('pickup_ts'))
print('Source:',SOURCE_ASSET,'sample limit:',MAX_ROWS)
# trip_id is a content fingerprint, NOT an official source/business trip identifier.
# Identical source rows intentionally have identical fingerprints.
df.show(5,truncate=False)


## Step 2 — Configure an explicit optional integration

Only Databricks asset rows are exported. The read is from that exported table, never a different dataset. This lab requires compatible compute, network access, a JDBC driver and organization-approved credentials.


In [ ]:
RUN_JDBC=False
JDBC_URL='' # Do not include credentials in this string.
JDBC_DRIVER='' # Example: com.ibm.db2.jcc.DB2Driver
JDBC_LAB_TABLE='' # A dedicated, pre-created course target that is empty before the first append.
SECRET_SCOPE=''; USER_KEY=''; PASSWORD_KEY=''
export=df.select('trip_id','pickup_zip','distance','fare').dropDuplicates(['trip_id']).limit(200)
print('JDBC enabled:',RUN_JDBC,'bounded rows:',export.count())


**Expected check:** No database connection or write occurs by default.


## Step 3 — Write and validate a course-only target

Append is not idempotent. Use an empty dedicated lab target or a database-side key/upsert contract before rerunning. The code refuses a nonempty target.


In [ ]:
if RUN_JDBC:
    import re
    assert JDBC_URL and JDBC_DRIVER and JDBC_LAB_TABLE and SECRET_SCOPE and USER_KEY and PASSWORD_KEY
    assert re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*',JDBC_LAB_TABLE), 'Use a simple dedicated schema.table target.'
    options={'url':JDBC_URL,'driver':JDBC_DRIVER,'user':dbutils.secrets.get(SECRET_SCOPE,USER_KEY),
             'password':dbutils.secrets.get(SECRET_SCOPE,PASSWORD_KEY),'dbtable':JDBC_LAB_TABLE}
    before=spark.read.format('jdbc').options(**options).load()
    assert before.limit(1).count()==0, 'Use an empty dedicated lab target; append retry can duplicate writes.'
    export.write.format('jdbc').options(**options).option('batchsize',200).option('numPartitions',2).mode('append').save()
    restored=spark.read.format('jdbc').options(**options).load()
    assert restored.count()==export.count()
    print('JDBC round-trip row count passed; inspect database type mappings as well.')
else:
    print('JDBC round trip skipped.')


**Expected check:** Enabled mode exports only course asset rows and reads only that same target.


## Step 4 — Understand bounds and parallelism

partitionColumn, lowerBound, upperBound and numPartitions must be supplied together for a partitioned JDBC read. Bounds calculate strides; they do not filter unwanted rows.


In [ ]:
print('Use a numeric/date/timestamp column with sensible distribution for partitionColumn.')
print('Lower/upper bounds and numPartitions define read strides; use an explicit predicate to restrict the dataset.')
print('A string trip_id cannot be used as Spark JDBC numeric partitionColumn.')
print('Write numPartitions limits concurrent connections; batchsize controls rows per JDBC batch.')


**Expected check:** The example explains the options without inventing an invalid partition column.


## Practice

Add a numeric IDENTITY column to the dedicated database lab target and use its actual MIN/MAX for a bounded parallel read.

## Solution guide

Use a safe course-only target, retrieve its typed bounds, and pass all four partition options. Preserve NULL semantics and verify precision/timestamp mappings. Spark retries can replay writes, so use database idempotency.

## Production considerations

Keep source lineage and stable business keys. Measure on representative volumes, review nulls and time semantics, and make retries idempotent. Sample-data checks demonstrate mechanics; they do not establish production performance or exactly-once external writes.
